## Filtering Operations on qdrant

In [1]:
import tqdm as notebook_tqdm

In [2]:
from sentence_transformers import SentenceTransformer

/home/saquib-siddiqui/tensorvault/learning/CB Ai Engineering Bootcamp/code_files/cb-ai-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from qdrant_client import QdrantClient, models
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)

# "path" = no server needed for demos
client = QdrantClient(url="http://localhost:6333")

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
DIM   = model.get_embedding_dimension()
print(f"Embedding dimension: {DIM}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3096.96it/s]


Embedding dimension: 384


In [5]:
COLLECTION_NAME = "my_collection"

def create_collection(name: str, distance: models.Distance):
    if not client.collection_exists(collection_name=name):
        client.create_collection(
            collection_name=name,
            vectors_config=models.VectorParams(size=DIM, distance=distance)
        )
        print(f"Collection '{name}' created successfully.")
    else:
        print(f"Collection '{name}' already exists.")

create_collection(COLLECTION_NAME, models.Distance.COSINE)

Collection 'my_collection' created successfully.


In [21]:
docs = ["Dogs are loyal and friendly domestic animals.",
        "Cats are independent and curious creatures.", 
        "The Milky Way galaxy contains over 200 billion stars.",
        "London is the capital city of England and the United Kingdom.",
        "The company made profits of $10 million in the last quarter.",
        "The stock market experienced a significant downturn due to economic uncertainty.",
        "The company plans to acquire a competitor for £45 million, but the transaction has not yet been publicly announced",
        "The CEO’s compensation package for the upcoming financial year includes a £2.5 million performance-related bonus tied to undisclosed strategic targets"
        ]

In [22]:
more_docs = ["The Great Wall of China is a historic fortification built to protect against invasions.",
            "The Amazon Rainforest is the largest tropical rainforest in the world, known for its biodiversity.",
            "The Eiffel Tower is an iconic landmark in Paris, France, and a symbol of architectural innovation.",
            "The Sahara Desert is the largest hot desert in the world, covering much of North Africa.",
            "The Taj Mahal is a UNESCO World Heritage Site and a symbol of love and architectural beauty in India.",
            "The Grand Canyon is a natural wonder in the United States, known for its breathtaking landscapes and geological formations.",
            "The Great Barrier Reef is the world's largest coral reef system, located off the coast of Australia and home to diverse marine life.",
            "The Pyramids of Giza in Egypt are ancient structures that have fascinated historians and archaeologists for centuries.",
            "Nuclear fusion is the process powering the sun.",
            "Quantum computing uses qubits to perform complex calculations.",
            "Machine learning enables computers to learn from data.",
            "Cats are curious and agile animals, often displaying playful behavior and independent personalities."]
docs.extend(more_docs)

In [23]:
documents = []

for i, doc in enumerate(docs):
    doc_id = i + 1
    doc_lower = doc.lower()

    # Confidential documents
    if (
        "plans to acquire" in doc_lower
        or "ceo’s compensation" in doc_lower
    ):
        category = "business"
        role = "admin"

    # Business documents
    elif any(word in doc_lower for word in [
        "company", "stock market", "profits"
    ]):
        category = "business"
        role = "employee"

    # Animals
    elif "cat" in doc_lower or "dog" in doc_lower:
        category = "animal"
        role = "employee"

    # Everything else
    else:
        category = "general"
        role = "employee"

    documents.append({
        "id": doc_id,
        "text": doc,
        "category": category,
        "role": role
    })

In [24]:
documents

[{'id': 1,
  'text': 'Dogs are loyal and friendly domestic animals.',
  'category': 'animal',
  'role': 'employee'},
 {'id': 2,
  'text': 'Cats are independent and curious creatures.',
  'category': 'animal',
  'role': 'employee'},
 {'id': 3,
  'text': 'The Milky Way galaxy contains over 200 billion stars.',
  'category': 'general',
  'role': 'employee'},
 {'id': 4,
  'text': 'London is the capital city of England and the United Kingdom.',
  'category': 'general',
  'role': 'employee'},
 {'id': 5,
  'text': 'The company made profits of $10 million in the last quarter.',
  'category': 'business',
  'role': 'employee'},
 {'id': 6,
  'text': 'The stock market experienced a significant downturn due to economic uncertainty.',
  'category': 'business',
  'role': 'employee'},
 {'id': 7,
  'text': 'The company plans to acquire a competitor for £45 million, but the transaction has not yet been publicly announced',
  'category': 'business',
  'role': 'admin'},
 {'id': 8,
  'text': 'The CEO’s com

In [27]:
texts    = [doc["text"] for doc in documents]
vectors  = model.encode(texts)

print(texts[0])
print(vectors[0])

Dogs are loyal and friendly domestic animals.
[-3.66339684e-02 -3.29479091e-02  2.27337386e-02  6.64364547e-02
 -5.36165833e-02  2.27458198e-02 -2.11714972e-02 -8.27348679e-02
  3.22351083e-02  4.99559864e-02  5.67642301e-02 -5.34846075e-02
  3.14257555e-02  4.75938655e-02  3.20098400e-02  2.32356619e-02
 -2.29435787e-02 -4.77748550e-02 -2.09068079e-02 -2.32110103e-03
 -1.37930796e-01 -2.33905949e-02  1.77905969e-02  7.32429209e-04
 -6.37915283e-02  1.69085748e-02  5.82159981e-02 -5.05699627e-02
  2.01622304e-02 -8.60782247e-03 -2.66483184e-02 -3.64649408e-02
  4.23132591e-02  1.81325041e-02 -7.42081702e-02  9.70803609e-04
 -5.28578786e-03  2.78006829e-02  1.63985956e-02  4.56886776e-02
  2.00761389e-02 -5.17774336e-02  6.47104606e-02 -7.60742724e-02
 -6.30132481e-02  2.71924138e-02 -9.39279273e-02 -2.33154632e-02
 -1.14535252e-02 -5.63323870e-02  1.98285785e-02  5.63527197e-02
  1.22258626e-03  1.21251224e-02  1.27016371e-02 -4.80140112e-02
 -2.83544697e-02 -2.07986366e-02  8.66122451

## Creating points with payload in qdrant

In [29]:
points = [
    models.PointStruct(
        id      = doc["id"],
        vector  = vectors[i].tolist(),
        payload = {
            "text"    : doc["text"],
            "category": doc["category"],
            "role"    : doc["role"],
        }
    )
    for i, doc in enumerate(documents)
]

In [30]:
operation_info = client.upsert(
    collection_name="my_collection",
    wait=True,
    points=points
)

print(f"Status: {operation_info.status}")

Status: completed


In [32]:
query     = "What animals make good pets?"
query_vec = model.encode(query).tolist()

results = client.query_points(
    collection_name="my_collection",
    query=query_vec,
    limit=3,
    #score_threshold=0.35
)

for r in results.points:
    print(f"Score: {r.score:.4f} | {r.payload['text']}")

Score: 0.5665 | Dogs are loyal and friendly domestic animals.
Score: 0.3954 | Cats are independent and curious creatures.
Score: 0.3900 | Cats are curious and agile animals, often displaying playful behavior and independent personalities.


In [34]:
query     = "What animals are curious, independent and agile?"
query_vec = model.encode(query).tolist()

results = client.query_points(
    collection_name="my_collection",
    query=query_vec,
    limit=3,
    #score_threshold=0.35
)

for r in results.points:
    print(f"Score: {r.score:.4f} | {r.payload['text']}")

Score: 0.6222 | Cats are curious and agile animals, often displaying playful behavior and independent personalities.
Score: 0.5158 | Cats are independent and curious creatures.
Score: 0.3939 | Dogs are loyal and friendly domestic animals.


## Payload Filtering

* `must` - AND
* `should` - OR
* `must_not` - AND NOT / NOT

In [36]:
documents[0]

{'id': 1,
 'text': 'Dogs are loyal and friendly domestic animals.',
 'category': 'animal',
 'role': 'employee'}

In [38]:
unique_categories = {doc["category"] for doc in documents}
unique_roles = {doc["role"] for doc in documents}

print("Categories:", unique_categories)
print("Roles:", unique_roles)

Categories: {'business', 'animal', 'general'}
Roles: {'admin', 'employee'}


In [43]:
results = client.query_points(
    collection_name="my_collection",
    query=model.encode("business is making millions of profits").tolist(),
    query_filter=Filter(
        must=[
            FieldCondition(
                key="category",
                match=MatchValue(value="business")
            )
        ]
    ),
    limit=3
)

for r in results.points:
    print(f"Score: {r.score:.4f} | {r.payload['text'][:60]}")

Score: 0.5169 | The company made profits of $10 million in the last quarter.
Score: 0.3221 | The company plans to acquire a competitor for £45 million, b
Score: 0.2346 | The CEO’s compensation package for the upcoming financial ye


In [45]:
results = client.query_points(
    collection_name="my_collection",
    query=model.encode("The business has made millions of profits and board is considering a raise for the CEO").tolist(),
    query_filter=Filter(
        must=[
            FieldCondition(key="category", match=MatchValue(value="business")),
            FieldCondition(key="role",     match=MatchValue(value="admin"))
        ]
    ),
    limit=5
)
for r in results.points:
    print(f"Score: {r.score:.4f} | {r.payload['text'][:60]}")

Score: 0.5263 | The CEO’s compensation package for the upcoming financial ye
Score: 0.4160 | The company plans to acquire a competitor for £45 million, b


In [47]:
results = client.query_points(
    collection_name="my_collection",
    query=model.encode("The business has made millions of profits and board is considering a raise for the CEO").tolist(),
    query_filter=Filter(
        must=[
            FieldCondition(key="category", match=MatchValue(value="business"))
        ],
        should=[
            FieldCondition(key="role", match=MatchValue(value="admin")),
            FieldCondition(key="role", match=MatchValue(value="employee"))
        ]
    ),
    limit=2
)

for r in results.points:
    print(f"Score: {r.score:.4f} | {r.payload['text']}")

Score: 0.5263 | The CEO’s compensation package for the upcoming financial year includes a £2.5 million performance-related bonus tied to undisclosed strategic targets
Score: 0.4441 | The company made profits of $10 million in the last quarter.
